# 🔍 Mini Project 3B — Pekan 2: Detector Anti-Pattern Gas
**Gas Optimization Framework untuk Smart Contract Solidity**

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Isi |
|---|---|
| 1 | Setup & import modul `src/` |
| 2 | Verifikasi AST parser |
| 3 | Demo semua 6 detector pada 1 kontrak |
| 4 | Batch run: semua detector × 50 kontrak |
| 5 | Statistik & agregat per domain |
| 6 | Simpan hasil ke `results/` |
| 7 | Hardhat benchmark anti-pattern |
| 8 | Checklist akhir Pekan 2 |

---
## ⚙️ Sel 1 — Setup & Import

In [1]:
import sys, json, subprocess, time
from pathlib import Path
from collections import defaultdict, Counter

ROOT        = Path().resolve().parent
HARDHAT_DIR = ROOT / 'hardhat_project'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ast_parser import generate_ast, walk_ast, find_nodes
from src.detectors import ALL_DETECTORS

# Load dari contracts_selection.json (dataset aktif)
# Ganti file ini untuk eksperimen dengan dataset berbeda
SELECTION_FILE = ROOT / 'contracts_selection.json'
if not SELECTION_FILE.exists():
    SELECTION_FILE = ROOT / 'contracts_metadata_expanded.json'
    print(f'⚠️  contracts_selection.json tidak ditemukan, fallback ke {SELECTION_FILE.name}')

with open(SELECTION_FILE) as f:
    metadata = json.load(f)

valid_contracts = [m for m in metadata if m.get('compile_ok', True)]

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.returncode != 0 and r.stderr.strip(): print('STDERR:', r.stderr.strip())
    return r

print('✅ Setup selesai')
print(f'   Sumber data       : {SELECTION_FILE.name}')
print(f'   Total metadata    : {len(metadata)}')
print(f'   Bisa compile      : {len(valid_contracts)}')
print(f'   Detektor tersedia : {len(ALL_DETECTORS)}')
for name, _ in ALL_DETECTORS:
    print(f'     • {name}')

✅ Setup selesai
   Sumber data       : contracts_selection.json
   Total metadata    : 75
   Bisa compile      : 74
   Detektor tersedia : 6
     • redundant_sload
     • unoptimized_loop
     • string_vs_bytes32
     • public_vs_external
     • unchecked_arithmetic
     • dead_code


---
## 🌳 Sel 2 — Verifikasi AST Parser

In [2]:
# Test pada kontrak pertama yang bisa compile
test_meta = valid_contracts[0]
ast = generate_ast(test_meta['file'])

if ast is None:
    print('❌ generate_ast() gagal')
else:
    funcs  = find_nodes(ast, 'FunctionDefinition')
    svars  = find_nodes(ast, 'StateVariableDeclaration')
    loops  = find_nodes(ast, 'ForStatement')

    print(f'✅ AST OK: {test_meta["nama"]} ({test_meta["domain"]})')
    print(f'   nodeType root      : {ast.get("nodeType")}')
    print(f'   FunctionDefinition : {len(funcs)}')
    print(f'   StateVariableDecl  : {len(svars)}')
    print(f'   ForStatement       : {len(loops)}')
    print()
    print('Contoh 3 fungsi pertama:')
    for fn in funcs[:3]:
        name = fn.get('name') or '(fallback/receive)'
        vis  = fn.get('visibility', '-')
        print(f'   {name:<30} visibility={vis}')

✅ AST OK: AppProxyUpgradeable (DeFi)
   nodeType root      : SourceUnit
   FunctionDefinition : 33
   StateVariableDecl  : 12
   ForStatement       : 0

Contoh 3 fungsi pertama:
   getStorageBool                 visibility=internal
   getStorageAddress              visibility=internal
   getStorageBytes32              visibility=internal


---
## 🔬 Sel 3 — Demo Semua 6 Detector pada 1 Kontrak

In [3]:
DEMO_FILE = 'Token_01_TetherToken.sol'   # ganti jika ingin kontrak lain

demo_meta = next(
    (m for m in valid_contracts if Path(m['file']).name == DEMO_FILE),
    valid_contracts[0]
)

print(f'📄 Kontrak demo : {demo_meta["nama"]}  ({demo_meta["domain"]}, {demo_meta["complexity"]})')
print(f'   File         : {Path(demo_meta["file"]).name}')
print(f'   LOC          : {demo_meta["loc"]}')
print()

demo_ast = generate_ast(demo_meta['file'])
if demo_ast is None:
    print('❌ AST None — skip demo')
else:
    print(f'{"Detector":<25} {"Temuan":>7}  Contoh Finding')
    print('─' * 80)
    for name, detect in ALL_DETECTORS:
        results = detect(demo_ast)
        contoh  = results[0]['description'][:60] + '...' if results else '(tidak ada temuan)'
        print(f'{name:<25} {len(results):>7}  {contoh}')

📄 Kontrak demo : AppProxyUpgradeable  (DeFi, Medium)
   File         : DeFi_08_AppProxyUpgradeable.sol
   LOC          : 279

Detector                   Temuan  Contoh Finding
────────────────────────────────────────────────────────────────────────────────
redundant_sload                 0  (tidak ada temuan)
unoptimized_loop                0  (tidak ada temuan)
string_vs_bytes32               0  (tidak ada temuan)
public_vs_external              8  Public vs External: fungsi 'hasPermission' dideklarasikan `p...
unchecked_arithmetic            0  (tidak ada temuan)
dead_code                       3  Dead Code: fungsi 'getStorageUint256' (internal) tidak perna...


---
## 🚀 Sel 4 — Batch Run: Semua Detector × 50 Kontrak
> Sel ini membutuhkan waktu 1–3 menit karena meng-compile semua kontrak.

In [4]:
print('⏳ Menjalankan semua detector pada semua kontrak...\n')

RESULTS = []
detector_names = [name for name, _ in ALL_DETECTORS]
total_n = len(metadata)

for i, m in enumerate(metadata, 1):
    entry = {
        'file'      : Path(m['file']).name,
        'nama'      : m['nama'],
        'domain'    : m['domain'],
        'complexity': m['complexity'],
        'loc'       : m['loc'],
        'compile_ok': m.get('compile_ok', False),
    }
    for dname in detector_names:
        entry[dname] = 0

    if not m.get('compile_ok', False):
        print(f'  [{i:3d}/{total_n}] ⚠️  SKIP  {m["nama"]}')
        RESULTS.append(entry)
        continue

    ast = generate_ast(m['file'])
    if ast is None:
        print(f'  [{i:3d}/{total_n}] ❌ AST None  {m["nama"]}')
        RESULTS.append(entry)
        continue

    total_findings = 0
    for dname, detect in ALL_DETECTORS:
        findings = detect(ast)
        entry[dname] = len(findings)
        total_findings += len(findings)

    print(f'  [{i:3d}/{total_n}] ✅  {m["nama"]:<35} total={total_findings:3d}')
    RESULTS.append(entry)

print(f'\n✅ Batch run selesai — {len(RESULTS)} kontrak diproses')

⏳ Menjalankan semua detector pada semua kontrak...

  [  1/75] ✅  AppProxyUpgradeable                 total= 11
  [  2/75] ✅  Dai                                 total= 10
  [  3/75] ✅  DaiJoin                             total=  2
  [  4/75] ✅  ETHJoin                             total=  2
  [  5/75] ✅  GemJoin                             total=  1
  [  6/75] ✅  KyberNetworkProxy                   total= 68
  [  7/75] ✅  Spotter                             total=  0
  [  8/75] ✅  UniswapV2Factory                    total= 20
  [  9/75] ✅  UniswapV2Pair                       total= 14
  [ 10/75] ✅  Vat                                 total= 24
  [ 11/75] ✅  CErc20                              total= 49
  [ 12/75] ✅  CErc20Delegator                     total=  6
  [ 13/75] ✅  CErc20Delegator                     total=  6
  [ 14/75] ✅  CEther                              total= 49
  [ 15/75] ✅  InitializableAdminUpgradeabilityProxy total=  0
  [ 16/75] ✅  CryptoPunksMarket               

---
## 📊 Sel 5 — Statistik & Agregat

In [5]:
compiled_results = [r for r in RESULTS if r['compile_ok']]

print('=' * 70)
print('  TOTAL TEMUAN PER DETECTOR')
print('=' * 70)
print(f'{"Detector":<28} {"Total":>7} {"Avg/kontrak":>12} {"Kontrak terdampak":>18}')
print('─' * 70)

for dname, _ in ALL_DETECTORS:
    total   = sum(r[dname] for r in compiled_results)
    avg     = total / len(compiled_results) if compiled_results else 0
    n_affected = sum(1 for r in compiled_results if r[dname] > 0)
    print(f'{dname:<28} {total:>7} {avg:>12.1f} {n_affected:>18}')

print('─' * 70)
grand_total = sum(r[dname] for r in compiled_results for dname, _ in ALL_DETECTORS)
print(f'{"TOTAL":<28} {grand_total:>7}')

print()
print('=' * 70)
print('  TEMUAN PER DOMAIN')
print('=' * 70)
header = f'{"Domain":<14}' + ''.join(f'{n[:7]:>9}' for n, _ in ALL_DETECTORS) + f'{"TOTAL":>8}'
print(header)
print('─' * 70)

domains = ['DeFi', 'NFT', 'Token', 'Governance', 'Utility']
for domain in domains:
    dr = [r for r in compiled_results if r['domain'] == domain]
    row = f'{domain:<14}'
    row_total = 0
    for dname, _ in ALL_DETECTORS:
        t = sum(r[dname] for r in dr)
        row += f'{t:>9}'
        row_total += t
    row += f'{row_total:>8}'
    print(row)

print()
print('📋 Top 10 kontrak dengan temuan terbanyak:')
print(f'{"Nama":<35} {"Domain":<12} {"Total":>6}')
print('─' * 55)
sorted_results = sorted(
    compiled_results,
    key=lambda r: sum(r[d] for d, _ in ALL_DETECTORS),
    reverse=True
)
for r in sorted_results[:10]:
    total = sum(r[d] for d, _ in ALL_DETECTORS)
    print(f'{r["nama"]:<35} {r["domain"]:<12} {total:>6}')

  TOTAL TEMUAN PER DETECTOR
Detector                       Total  Avg/kontrak  Kontrak terdampak
──────────────────────────────────────────────────────────────────────
redundant_sload                  634          8.6                 47
unoptimized_loop                   7          0.1                  2
string_vs_bytes32                243          3.3                 44
public_vs_external               649          8.8                 58
unchecked_arithmetic               0          0.0                  0
dead_code                        122          1.6                 26
──────────────────────────────────────────────────────────────────────
TOTAL                           1655

  TEMUAN PER DOMAIN
Domain          redunda  unoptim  string_  public_  uncheck  dead_co   TOTAL
──────────────────────────────────────────────────────────────────────
DeFi                151        2       24       56        0       29     262
NFT                 208        0      112      183        0     

---
## 💾 Sel 6 — Simpan Hasil ke `results/`

In [6]:
import csv

# ── Simpan JSON lengkap
json_path = RESULTS_DIR / 'pekan2_detector_results.json'
with open(json_path, 'w') as f:
    json.dump(RESULTS, f, indent=2)
print(f'✅ JSON disimpan : {json_path}')

# ── Simpan CSV ringkasan
csv_path = RESULTS_DIR / 'pekan2_summary.csv'
fieldnames = ['nama', 'domain', 'complexity', 'loc', 'compile_ok'] + \
             [name for name, _ in ALL_DETECTORS] + ['total_findings']

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    for r in RESULTS:
        row = dict(r)
        row['total_findings'] = sum(r.get(d, 0) for d, _ in ALL_DETECTORS)
        writer.writerow(row)

print(f'✅ CSV disimpan  : {csv_path}')

# ── Preview tabel
print()
print(f'{"Nama":<35} {"Domain":<12} {"LOC":>5} {"OK?":>4} ', end='')
print('  '.join(f'{n[:4]:>4}' for n, _ in ALL_DETECTORS))
print('─' * 90)
for r in RESULTS:
    ok = '✅' if r['compile_ok'] else '❌'
    row = f'{r["nama"]:<35} {r["domain"]:<12} {r["loc"]:>5} {ok:>4}   '
    row += '    '.join(f'{r.get(d, 0):>4}' for d, _ in ALL_DETECTORS)
    print(row)

✅ JSON disimpan : C:\Users\Ridho\project\KBJ\gas_optimization\results\pekan2_detector_results.json
✅ CSV disimpan  : C:\Users\Ridho\project\KBJ\gas_optimization\results\pekan2_summary.csv

Nama                                Domain         LOC  OK? redu  unop  stri  publ  unch  dead
──────────────────────────────────────────────────────────────────────────────────────────
AppProxyUpgradeable                 DeFi           279    ✅      0       0       0       8       0       3
Dai                                 DeFi           169    ✅      7       0       3       0       0       0
DaiJoin                             DeFi           192    ✅      1       0       0       1       0       0
ETHJoin                             DeFi           192    ✅      1       0       0       1       0       0
GemJoin                             DeFi           162    ✅      1       0       0       0       0       0
KyberNetworkProxy                   DeFi           484    ✅     24       2       0      40

---
## ⛽ Sel 7 — Hardhat Benchmark Anti-Pattern
Mengukur penghematan gas empiris untuk anti-pattern **Redundant SLOAD**.
Kontrak `BenchmarkSLOAD.sol` mensimulasikan kasus nyata.

In [7]:
# ── Tulis kontrak benchmark
bench_sol = """// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract BenchmarkSLOAD {
    uint256 public counter;
    uint256 public multiplier;

    constructor() { counter = 10; multiplier = 3; }

    // Anti-pattern: 3x SLOAD (counter dibaca 3x dari storage)
    function computeBoros() external view returns (uint256) {
        return counter * counter + counter;  // 3 SLOAD
    }

    // Optimal: 1x SLOAD + cache lokal
    function computeHemat() external view returns (uint256) {
        uint256 c = counter;  // 1 SLOAD
        return c * c + c;
    }

    // Anti-pattern: loop membaca .length dari storage setiap iterasi
    uint256[] public items;
    function sumBoros() external view returns (uint256 total) {
        for (uint256 i = 0; i < items.length; i++) {  // items.length = SLOAD tiap iterasi
            total += items[i];
        }
    }

    // Optimal: cache length sebelum loop
    function sumHemat() external view returns (uint256 total) {
        uint256 len = items.length;  // 1 SLOAD
        for (uint256 i = 0; i < len; i++) {
            total += items[i];
        }
    }

    function populateItems(uint256 n) external {
        for (uint256 i = 0; i < n; i++) items.push(i + 1);
    }
}
"""

# ── Script JS benchmark
bench_js = """const { ethers } = require("hardhat");

async function main() {
  const F = await ethers.getContractFactory("BenchmarkSLOAD");
  const c = await F.deploy();
  await c.waitForDeployment();

  // Populate 10 items
  const tx0 = await c.populateItems(10); await tx0.wait();

  // Ukur computeBoros vs computeHemat
  const g1 = await c.computeBoros.estimateGas();
  const g2 = await c.computeHemat.estimateGas();
  const d1 = g1 - g2;
  const p1 = (Number(d1) * 100 / Number(g1)).toFixed(2);

  // Ukur sumBoros vs sumHemat
  const g3 = await c.sumBoros.estimateGas();
  const g4 = await c.sumHemat.estimateGas();
  const d2 = g3 - g4;
  const p2 = (Number(d2) * 100 / Number(g3)).toFixed(2);

  console.log("=== BENCHMARK ANTI-PATTERN GAS ===");
  console.log("[Redundant SLOAD — compute]");
  console.log("  computeBoros :", g1.toString(), "gas");
  console.log("  computeHemat :", g2.toString(), "gas");
  console.log("  Selisih      :", d1.toString(), "gas ("+p1+"%)");
  console.log("");
  console.log("[Unoptimized Loop — sum(10 items)]");
  console.log("  sumBoros     :", g3.toString(), "gas");
  console.log("  sumHemat     :", g4.toString(), "gas");
  console.log("  Selisih      :", d2.toString(), "gas ("+p2+"%)");
}
main().catch(e => { console.error(e); process.exitCode = 1; });
"""

(HARDHAT_DIR / 'contracts' / 'BenchmarkSLOAD.sol').write_text(bench_sol)
(HARDHAT_DIR / 'scripts'   / 'benchmark_sload.js').write_text(bench_js)
print('✅ BenchmarkSLOAD.sol & benchmark_sload.js ditulis')

run('npx hardhat compile', cwd=HARDHAT_DIR)
print()
run('npx hardhat run scripts/benchmark_sload.js --network hardhat', cwd=HARDHAT_DIR)

✅ BenchmarkSLOAD.sol & benchmark_sload.js ditulis
Nothing to compile

=== BENCHMARK ANTI-PATTERN GAS ===
[Redundant SLOAD ï¿½ compute]
  computeBoros : 24186 gas
  computeHemat : 23955 gas
  Selisih      : 231 gas (0.96%)

[Unoptimized Loop ï¿½ sum(10 items)]
  sumBoros     : 51208 gas
  sumHemat     : 50088 gas
  Selisih      : 1120 gas (2.19%)


CompletedProcess(args='npx hardhat run scripts/benchmark_sload.js --network hardhat', returncode=0, stdout='=== BENCHMARK ANTI-PATTERN GAS ===\n[Redundant SLOAD ï¿½ compute]\n  computeBoros : 24186 gas\n  computeHemat : 23955 gas\n  Selisih      : 231 gas (0.96%)\n\n[Unoptimized Loop ï¿½ sum(10 items)]\n  sumBoros     : 51208 gas\n  sumHemat     : 50088 gas\n  Selisih      : 1120 gas (2.19%)\n', stderr='')

---
## ✅ Sel 8 — Checklist Akhir Pekan 2

In [8]:
n_compiled   = sum(1 for r in RESULTS if r['compile_ok'])
n_any_finding = sum(1 for r in RESULTS if r['compile_ok'] and
                    sum(r.get(d, 0) for d, _ in ALL_DETECTORS) > 0)
total_findings = sum(r.get(d, 0) for r in RESULTS for d, _ in ALL_DETECTORS)

checks = [
    ('src/ast_parser.py ada',
        (ROOT / 'src' / 'ast_parser.py').exists()),
    ('6 modul detector ada',
        len(list((ROOT / 'src' / 'detectors').glob('*.py'))) >= 7),  # 6 + __init__
    ('ALL_DETECTORS berisi 6 item',
        len(ALL_DETECTORS) == 6),
    ('Minimal 40 kontrak di-compile',
        n_compiled >= 40),
    ('Semua 6 detector berjalan tanpa error',
        True),   # jika Sel 4 selesai tanpa exception
    ('Minimal 1 finding ditemukan',
        total_findings > 0),
    ('Minimal 20 kontrak ada temuan',
        n_any_finding >= 20),
    ('results/pekan2_summary.csv tersimpan',
        (RESULTS_DIR / 'pekan2_summary.csv').exists()),
    ('Hardhat benchmark selesai',
        (HARDHAT_DIR / 'contracts' / 'BenchmarkSLOAD.sol').exists()),
]

print('=' * 50)
print('  CHECKLIST AKHIR PEKAN 2')
print('=' * 50)

semua_ok = True
for label, ok in checks:
    icon = '✅' if ok else '❌'
    if not ok: semua_ok = False
    print(f'  {icon} {label}')

print('─' * 50)
print(f'  Kontrak dianalisis : {n_compiled}')
print(f'  Total findings     : {total_findings}')
print(f'  Kontrak bertanda   : {n_any_finding}')
print('=' * 50)
print('  🎉 Siap masuk Pekan 3!' if semua_ok
      else '  ⚠️  Cek item ❌ di atas')
print('=' * 50)

  CHECKLIST AKHIR PEKAN 2
  ✅ src/ast_parser.py ada
  ✅ 6 modul detector ada
  ✅ ALL_DETECTORS berisi 6 item
  ✅ Minimal 40 kontrak di-compile
  ✅ Semua 6 detector berjalan tanpa error
  ✅ Minimal 1 finding ditemukan
  ✅ Minimal 20 kontrak ada temuan
  ✅ results/pekan2_summary.csv tersimpan
  ✅ Hardhat benchmark selesai
──────────────────────────────────────────────────
  Kontrak dianalisis : 74
  Total findings     : 1655
  Kontrak bertanda   : 66
  🎉 Siap masuk Pekan 3!
